In [ ]:
# Для установки библиотеки воспользуйтесь командой
# !conda install conda-forge::transformers
# или
# !pip install transformers

In [ ]:
# Любая LLM работает с токенами, поэтому нам нужна не только модель, но и токенизатор
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Зададим название модели из репозитория huggingface
# список доступных моделей можно посмотреть по ссылке https://huggingface.co/models
# будем использовать модель Qwen или YandexGPT

model_name = "Qwen/Qwen2.5-7B-Instruct-1M"
#model_name = "yandex/YandexGPT-5-Lite-8B-instruct"

# В названии модели обычно указываются - версия (2,5) количество параметров (7 млрд.)
# этап обучения или предназначение (Base, Instruct, Code, Thinking и т.п.)
# размер контекстного окна (1 млн.)
# также могут указываться параметры квантования (FP8, GGUF, ) и другие характеристики.


# Загружаем предобученную можель
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# Загружаем предобученный токенизатор
tokenizer = AutoTokenizer.from_pretrained(model_name)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Загружаем текстовый файл преобразуя его в строку
file = 'ENG_article.txt'
with open(file, 'r', encoding='cp1251') as file:
    data = file.read().replace('\n', ' ')

print(len(data))

8291


In [ ]:
# Посмотрим выдержку из файла
print(data[90:160])

 (machine learning)     This article is about the computational models


In [ ]:
# Посчитаем количество слов в файле чтобы примерно сопоставить с контекстным окном
num_words = len(data.split())
print(num_words)

1157


In [ ]:
# Для общения с LLM нужно составить промпт
# он будет состоять из системной части - наши инструкции что нужно сделать
# и пользовательской части - текста файла
messages = [
    {"role": "system", "content": "Проанализируй текст. После ответь на 3 вопроса: -	В каком году была обозначена проблема взрывающихся градиентов?	Кто в 1891 году разработал метод уничтожающей производной? Кто предложил цепное правило дифференцирования и в каком году? Ответ давай на русском языке."},
    {"role": "user", "content": data} # берем только часть чтобы долго не ждать генерацию
]

# Для YandexGPT инструкции пишутся от имени пользователя
#messages = [
#    {"role": "user", "content": "Суммаризируй текст. Определи жанр текста, выдели основную информацию. Ответ давай на русском языке."+data[0:5000]} # берем только часть чтобы долго не ждать генерацию
#]

# Метод apply_chat_template() используется для форматирования сообщений в одну строку, формат
# которой ориентирован на использование чат-ориентированных языковых моделей.
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# В текст добавляются специальные  токены-метки для указания структуры разговора.
print(text[0:160])

<|im_start|>system
Проанализируй текст. После ответь на 3 вопроса: -	В каком году была обозначена проблема взрывающихся градиентов?	Кто в 1891 году разработал м


In [ ]:
# Токенизатор разбивает текст на токены.
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# input_ids — токены в числовом виде, которые представляют собой уникальный номер (ID) из словаря модели.
# attention_mask — задает маску, которая показывает позиции реальных токенов и дополнений (padding) или служебных токенов.
# Модели обрабатывают батчи текстов одинаковой длины. Чтобы выровнять длину реальных текстов короткие тексты дополняются специальным токеном [PAD].
# Таким токенам в attention_mask соответствуют нули, чтобы модель их не обрабатывала.
print(model_inputs)

{'input_ids': tensor([[151644,   8948,    198,  ..., 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0')}


In [ ]:
# Токенизированный текст подаем в модель.
# max_new_tokens - задает максимальное число генерируемых в ответе токенов.
# Можно указать и другие параметры:
# temperature - регулирует "случайность" в выборе следующего токена (<1 — более предсказуемо, >1 — более хаотично).
# top_k - ограничивает выбор следующего токена K наиболее вероятными
# repetition_penalty - штраф за повторяющиеся токены (>1 - штраф, 1 - без штрафа).
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=1024,
#    temperature=0.9,
#    top_k=50,
#    repetition_penalty=1.2
)

In [ ]:
# Так как модель возвращает и промпт и сгенерированные токены, выделяем только ответ.
generated_ids_ = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# Преобразуем ID токенов обратно в слова.
response = tokenizer.batch_decode(generated_ids_, skip_special_tokens=True)[0]

In [ ]:
# Смотрим что получилось.
print(response)

Вопросы:

1. В каком году была обозначена проблема взрывающихся градиентов?
Ответ: В 1991 году.

2. Кто в 1891 году разработал метод уничтожающей производной?
Ответ: Метод уничтожающей производной не был разработан в 1891 году. Это понятие связано с математическим анализом и было развито другими учеными, но в указанном источнике конкретного упоминания этого метода за 1891 год нет.

3. Кто предложил цепное правило дифференцирования и в каком году?
Ответ: Цепное правило дифференцирования было предложено Готфридом Лейбницем в 1673 году.
